In [3]:
!pip install -q websockets asyncio pandas folium

In [ ]:
import asyncio
import websockets
import json
from datetime import datetime
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
import folium
from folium import plugins

# --- Configuration ---
API_KEY = "'your-ais-api-key'" #aisstream.io - get the API key from your account dashboard
PROJECT_ID = "your-gcp-project-id" 
DATASET_ID = f"{PROJECT_ID}.hormuz_conflict_data"
TABLE_ID = f"{DATASET_ID}.live_vessel_tracking"

# Bounding Box for Strait of Hormuz / Gulf of Oman
BOUNDING_BOX = [[[24.0, 54.0], [27.0, 57.0]]]
#BOUNDING_BOX = [[[25.8, 55.2], [27.2, 56.8]]] #tighter box, if you want to focus on the central area

bq_client = bigquery.Client(project=PROJECT_ID)



In [ ]:
def create_visualizer(bbox_data):
    # 1. Extract coordinates from the nested list format
    southwest = bbox_data[0][0]
    northeast = bbox_data[0][1]

    # Calculate the center point to focus the map
    center_lat = (southwest[0] + northeast[0]) / 2.0
    center_lon = (southwest[1] + northeast[1]) / 2.0

    # 2. Create the base map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7, control_scale=True)

    # Add alternative map layers for better geography visualization
    folium.TileLayer('cartodbpositron', name='Light Map').add_to(m)
    folium.TileLayer('OpenTopoMap', name='Topographic Map').add_to(m)

    # 3. Add the Bounding Box
    folium.Rectangle(
        bounds=[southwest, northeast],
        color='#cc0000',          # Red border
        weight=3,                 # Border thickness
        fill=True,
        fill_color='#ffcccc',     # Light red fill
        fill_opacity=0.3,
        tooltip=f"<strong>Bounding Box</strong><br>SW: {southwest}<br>NE: {northeast}"
    ).add_to(m)

    # 4. Add key landmarks for geographic context and just because we can
    landmarks = {
        'Dubai, UAE': [25.2048, 55.2708],
        'Abu Dhabi, UAE': [24.4539, 54.3773],
        'Bandar Abbas, Iran': [27.1832, 56.2666],
        'Khasab, Oman': [26.1833, 56.2500],
        'Strait of Hormuz (Choke Point)': [26.5667, 56.2500]
    }

    for name, coords in landmarks.items():
        icon_color = 'red' if 'Strait' in name else 'blue'
        icon_type = 'star' if 'Strait' in name else 'info-sign'

        folium.Marker(
            location=coords,
            popup=f"<b>{name}</b><br>Lat: {coords[0]}<br>Lon: {coords[1]}",
            tooltip=f"Click for info: {name}",
            icon=folium.Icon(color=icon_color, icon=icon_type)
        ).add_to(m)

    # 5. Add Interactive Plugins
    plugins.Fullscreen(position='topright', title='Expand', title_cancel='Exit', force_separate_button=True).add_to(m)
    minimap = plugins.MiniMap(toggle_display=True, position='bottomright')
    m.add_child(minimap)
    m.add_child(plugins.MeasureControl(position='bottomleft', primary_length_unit='kilometers'))

    # 6. Add Layer Control
    folium.LayerControl().add_to(m)

    # 7. Return the map object so the notebook renders it inline
    return m

create_visualizer(BOUNDING_BOX)

In [ ]:
def setup_bigquery():
    """Ensures the Dataset and Table exist before streaming begins."""
    dataset = bigquery.Dataset(DATASET_ID)
    dataset.location = "US"
    try:
        bq_client.get_dataset(dataset)
    except NotFound:
        bq_client.create_dataset(dataset, timeout=30)
        print(f"Created dataset {DATASET_ID}.")

    schema = [
        bigquery.SchemaField("timestamp", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("mmsi", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("ship_name", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("latitude", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("longitude", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("speed_over_ground", "FLOAT", mode="NULLABLE"),
    ]

    table = bigquery.Table(TABLE_ID, schema=schema)
    try:
        bq_client.get_table(table)
    except NotFound:
        bq_client.create_table(table, timeout=30)
        print(f"Created table {TABLE_ID}.")

async def stream_ais_to_bq():
    """Connects to WebSocket and streams rows directly to BigQuery."""
    # Fixed syntax: "APIKey" without the underscore
    subscribe_message = {
        "APIKey": API_KEY,
        "BoundingBoxes": BOUNDING_BOX,
        "FilterMessageTypes": ["PositionReport"]
    }

    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as websocket:
        await websocket.send(json.dumps(subscribe_message))
        print(f"[{datetime.utcnow().isoformat()}] Live stream connected. Ingesting to BigQuery...")

        try:
            async for message_json in websocket:
                try:
                    message = json.loads(message_json)

                    # Defensive check for MessageType
                    if message.get("MessageType") == "PositionReport":
                        ais = message.get("MetaData", {})
                        pos = message.get("Message", {}).get("PositionReport", {})

                        if not ais or not pos:
                            continue

                        # --- Timestamp Parsing for BigQuery ---
                        raw_ts = ais.get("time_utc", "")
                        if raw_ts:
                            base_ts = raw_ts.split(' +')[0] # Remove ' +0000 UTC'
                            if '.' in base_ts:
                                dt_part, frac = base_ts.split('.')
                                clean_ts = f"{dt_part}.{frac[:6]}" # Max 6 decimal places for microseconds
                            else:
                                clean_ts = base_ts
                        else:
                            clean_ts = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S.%f')

                        row = [{
                            "timestamp": clean_ts,
                            "mmsi": int(ais.get("MMSI", 0)),
                            "ship_name": ais.get("ShipName", "").strip(),
                            "latitude": float(pos.get("Latitude", 0.0)),
                            "longitude": float(pos.get("Longitude", 0.0)),
                            "speed_over_ground": float(pos.get("Sog", 0.0))
                        }]

                        errors = bq_client.insert_rows_json(TABLE_ID, row)
                        if errors:
                            print(f"\nInsert errors: {errors}")
                        else:
                            # Print a dot for every successful insertion so you know it is working
                            print(".", end="", flush=True)

                except json.JSONDecodeError:
                    pass
                except Exception as e:
                    print(f"\nRow processing error: {e}")

        except websockets.exceptions.ConnectionClosedError:
            print("\nConnection dropped by server. Restarting cell is recommended.")

# Execute setup, then run the async loop
setup_bigquery()
await stream_ais_to_bq()

[2026-03-16T12:49:02.381990] Live stream connected. Ingesting to BigQuery...
...........................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [ ]:
#BIGQUERY SAMPLE QUERY to DEDUPE AND GET LATEST POSITION PER VESSEL:

WITH LatestVesselPings AS (
  SELECT 
    mmsi,
    ship_name,
    latitude,
    longitude,
    speed_over_ground,
    timestamp,
    -- This window function assigns a '1' to the absolute newest ping for each ship
    ROW_NUMBER() OVER(PARTITION BY mmsi ORDER BY timestamp DESC) as ping_rank
  FROM 
    `hormuz_conflict_data.live_vessel_tracking`
  WHERE
    -- Filter out GPS anomalies (coordinates outside our bounding box)
    latitude BETWEEN 24.0 AND 27.0
    AND longitude BETWEEN 54.0 AND 57.0
)

SELECT 
  mmsi,
  -- Clean up empty ship names
  COALESCE(NULLIF(ship_name, ''), 'UNKNOWN_VESSEL') AS vessel_name,
  latitude,
  longitude,
  speed_over_ground AS speed_knots,
  timestamp AS last_seen_utc,
  
  -- Tactical Classification Engine
  CASE 
    WHEN speed_over_ground < 0.5 THEN 'Ghost Fleet (Anchored/Paralyzed)'
    WHEN speed_over_ground BETWEEN 0.5 AND 3.5 THEN 'Caution (Creeping)'
    ELSE 'Active Transit (Underway)'
  END AS tactical_status,

  -- Generate a native GIS point for Looker Studio / Kepler.gl
  ST_GEOGPOINT(longitude, latitude) AS spatial_point
  

FROM 
  LatestVesselPings
WHERE 
  ping_rank = 1 -- Only pull the current real-time state of the board
ORDER BY 
  speed_knots ASC;
